# BusState Data Processing Pipeline (Python)

### Overview

This notebook processes raw **BusState APC data** from compressed `.txt.zip` files into clean, structured datasets for downstream analysis and dashboarding.

The workflow is a Python translation of an existing R-based pipeline previously used in the OSU Department of Transportation and Traffic Management (TTM), with improvements for modularity, readability, and scalability.

---

### Data Source

* Directory: `K:/AP/TTM/Data/APC Data/`
* File format: `compressed .txt`
* Naming convention:

  ```
  busstate0####DDMMYY.txt.zip
  ```

  * `####` = bus identifier
  * `DD` = day
  * `MM` = month
  * `YY` = year (2-digit)

---

### Output (`sort_and_save`)

* Data is split into monthly subsets

* Saved as `.csv` files:

  ```
  YYYY-MMM-busstate.csv
  ```

* Example:

  ```
  2025-OCT-busstate.csv
  ```

* Output directory:

  ```
  K:/AP/TTM/Data/WMC Dashboard/BusState Cleaned/
  ```

---

### Notes

* This will take around 3-4 minutes to run for one month worth of data.
* If this is run on a Unix based machine, changes will need to be made to the root directory.
* Potential issues with overwriting in the sort_and_save function when running a new month with existing data. - will revisit

---

### Author

* Writen by Clayton Morgan (morgan.1461) Reporting and Analytics Analyst at The Ohio State University
* Python implementation of legacy R workflow in the TTM


In [1]:
import busstate_processing as bp
import dashboard_processing as dp
# note for future me: if you edit processing.py, run the lines below to reload w/out restarting kernel

# import importlib
# importlib.reload(bp)
# importlib.reload(dp)

**Note**: The year and month fields must be in 2 digit format, within the quotations.

e.g) For October, 2025

year = "25"

month = "09"

In [2]:
# STEP 1: Set month and year of interest for dashboard
year = "25"
month = "12"

In [3]:
# STEP 2: Run processing pipeline to clean busstate data for month and year of interest
bp.busstate_processing(year, month)

Starting busstate processing for 12/25...
Found 902 busstate files for 12/25 in 'K:/AP/TTM/Data/APC Data'
Unzipping and processing busstate files for 12/25...
Combined dataframe has 868244 records for 12/25
No data for month JAN 2025
No data for month FEB 2025
No data for month MAR 2025
No data for month APR 2025
No data for month MAY 2025
No data for month JUN 2025
No data for month JUL 2025
No data for month AUG 2025
Saved 9 records for SEP 2025 to 'K:\AP\TTM\Data\WMC Dashboard\BusState Cleaned\2025-SEP-busstate.csv'
No data for month OCT 2025
Saved 26107 records for NOV 2025 to 'K:\AP\TTM\Data\WMC Dashboard\BusState Cleaned\2025-NOV-busstate.csv'
Saved 842127 records for DEC 2025 to 'K:\AP\TTM\Data\WMC Dashboard\BusState Cleaned\2025-DEC-busstate.csv'
Finished processing busstate data for 12/25 in 180.94 seconds. Cleaned files saved to 'K:/AP/TTM/Data/WMC Dashboard/BusState Cleaned'


### Stops inventory and metrics
This code will produce a dataframe containing the stops including the new University Hopsital and Doan Hall stops.

In [30]:
import pandas as pd
import numpy as np
import os
import pathlib

In [18]:
def build_stops_df():
    '''
    Build stops dataframe by merging static pattern stops and stop inventory files. Anytime that stops change, this will need to be re run after the csv files are updated.

    Parameters:
        None
    Returns:
        stops_df (pd.DataFrame): dataframe with stop id, stop name, and lat/lon coordinates for all stops in the pattern stops file
    '''
    # set directory paths for stop data - stored as 2 csv files in ./stops/
    stop_data_dir = pathlib.Path(os.getcwd()) / "stops"
    pattern_stops_path = stop_data_dir / "pattern_stops.csv"
    stop_inventory_path = stop_data_dir / "stop_inventory.csv"

    # read in static stop files
    pattern_stops = pd.read_csv(pattern_stops_path, header = None) # no header
    stop_inventory = pd.read_csv(stop_inventory_path)

    # only need cols 0 and 9 and can drop any dups
    pattern_stops = pattern_stops.iloc[:, [0, 9]].drop_duplicates()
    pattern_stops.columns = ["ROUTE", "STOP_ID"] # rename cols for merge

    # Merge pattern stops and stop inventory on stop id - left merge 
    stops_df = pattern_stops.merge(stop_inventory, how = "left", on = "STOP_ID")

    return stops_df

In [101]:
def which_stop(lat, lon, route = "MC", max_distance = 0.0005):
    '''
    Determine the closest stop to a given latitude and longitude. Med center route is default, but can be updated to other routes as needed. 
    NOTE: stops_df must be a global variable for this function to work, so build_stops_df() must be run before this function is called.

    Args:
        lat (float): The latitude of the point of interest.
        lon (float): The longitude of the point of interest.
        route (str): The route to filter stops by. Default is "MC" for medical center. Based on ROUTE col in stops DataFrame. Potentially important in future for overlapping stops.
        max_distance (float): The maximum distance to consider for a stop. Default is 0.0005 per legacy R code. approximately 150ish feet

    Returns:
        int: The STOP_ID of the closest stop.
    '''
    # subset the stops to route of interest
    route_stops = stops_df[stops_df['ROUTE'] == route]

    distance_lon = route_stops['LONG'].values - lon
    distance_lat = route_stops['LAT'].values - lat

    # calculate the distance to each stop
    distances = np.sqrt(distance_lon**2 + distance_lat**2)

    mask = distances < max_distance
    # lowest distance should be the closest stop now. - each stop is at minimum approx 430ft apart so margin of 150 should be fine for MC route.
    selected_stop = route_stops[mask]

    # if no stops within alloted distance, return None
    if not np.any(mask):
        return None

    return int(route_stops.loc[mask, 'STOP_ID'].iloc[0])


In [173]:
# NOTE: Will need to update function with year and month functionality.
def process_mc_busstate():
    """
    Process the busstate data for the medical center route.
    NOTE: This will ONLY work for the medical center route.
    NOTE: This will return a significantly smaller dataframe
    Args:
        None
    Returns:
        DataFrame: A pandas DataFrame containing the processed busstate data for the medical center route.
    """
    year_full = "2025" # add function to convert, etc
    month_full = "DEC"

    busstate_dir = pathlib.Path(os.getcwd()) / "BusState Cleaned"
    busstate_path = os.path.normpath(os.path.join(busstate_dir, f"{year_full}-{month_full}-busstate.csv"))
    busstate_df = pd.read_csv(busstate_path) # Now cleaned busstate data read in

    # Process the busstate data for medical center routes
    #busstate_df.info()

    # should filter to just MC routes
    filtered_busstate_df = busstate_df.loc[(busstate_df['RUN_ID'] > 1500) & (busstate_df['RUN_ID'] < 1600)].copy()

    # Convert time metrics to datetime
    filtered_busstate_df['EVENT_TIME'] = pd.to_datetime(filtered_busstate_df['EVENT_TIME'], format = "%H:%M:%S")
    filtered_busstate_df['DEPARTURE_TIME'] = pd.to_datetime(filtered_busstate_df['DEPARTURE_TIME'], format = "%H:%M:%S")
    filtered_busstate_df['ENTER_STOP_WINDOW_TIME'] = pd.to_datetime(filtered_busstate_df['ENTER_STOP_WINDOW_TIME'], format = "%H:%M:%S")
    filtered_busstate_df['EXIT_STOP_WINDOW_TIME'] = pd.to_datetime(filtered_busstate_df['EXIT_STOP_WINDOW_TIME'], format = "%H:%M:%S")

    # sort by date and event time
    filtered_busstate_df = filtered_busstate_df.sort_values(['DATE', 'EVENT_TIME'])

    # Assign stop ID - most resource intensive step - as INT not float
    filtered_busstate_df['STOP_ID'] = filtered_busstate_df.apply(lambda row: which_stop(row['LATITUDE'], row['LONGITUDE']), axis = 1)

    # filter out 'dummy' stops, 27 and 461
    filtered_busstate_df = filtered_busstate_df.drop(filtered_busstate_df[filtered_busstate_df['STOP_ID'].isin([27, 461])].index)

    # filter out empty stops - where bus was on route and not at a stop geographically
    filtered_busstate_df = filtered_busstate_df.drop(filtered_busstate_df[filtered_busstate_df['STOP_ID'].isna()].index)

    # Convert to int
    filtered_busstate_df['STOP_ID'] = filtered_busstate_df['STOP_ID'].astype(int)

    # filter by bus id, date, event time
    filtered_busstate_df = filtered_busstate_df.sort_values(['BUS_ID', 'DATE', 'EVENT_TIME']).reset_index(drop = True) # restting index

    # create a new col for each new event
    # when stop changes OR bus_id changes OR elapsed time > 60
    count = (
        (filtered_busstate_df['STOP_ID'] != filtered_busstate_df['STOP_ID'].shift(1)) |
        (filtered_busstate_df['BUS_ID'] != filtered_busstate_df['BUS_ID'].shift(1)) |
        ((filtered_busstate_df['EVENT_TIME'] - filtered_busstate_df['EVENT_TIME'].shift(1)).dt.total_seconds() > 60)
    )

    # take cumsum of count to assign 
    filtered_busstate_df['COUNT'] = count.cumsum()

    # consolidate df
    consolidated_busstate_df = filtered_busstate_df.groupby(['BUS_ID', 'DATE', 'COUNT', 'STOP_ID', 'RUN_ID'], as_index = False).agg(
        BOARDINGS = ('BOARDINGS', 'max'),
        ALIGHTINGS = ('ALIGHTINGS', 'max'),
        LOAD = ('PASSENGER_LOAD', 'max'),
        EARLY_EVENT=("EVENT_TIME", "min"),
        LATE_EVENT=("EVENT_TIME", "max"),
        DEPARTURE_TIME=("DEPARTURE_TIME", "max"),
        ENTER_STOP=("ENTER_STOP_WINDOW_TIME", "min"),
        EXIT_STOP=("EXIT_STOP_WINDOW_TIME", "max"),
        RUN_ID = ("RUN_ID", "last"),
        DEST=("DEST_SIGN_ROUTE_TEXT", "last"),
    )
    #print(consolidated_busstate_df)

    # calculate the arrival and departure times for each event
    consolidated_busstate_df['ARRIVAL'] = consolidated_busstate_df[['EARLY_EVENT', 'ENTER_STOP']].min(axis=1) # arrival is earlier enter stop or early event
    consolidated_busstate_df['DEPARTURE'] = consolidated_busstate_df[['LATE_EVENT', 'DEPARTURE_TIME', 'EXIT_STOP']].max(axis=1) # departure is latest of late event or departure time or exit stop
    consolidated_busstate_df['DWELL'] = (consolidated_busstate_df['DEPARTURE'] - consolidated_busstate_df['ARRIVAL']) # total dwell time at stop

    # can drop unnecessary cols now
    consolidated_busstate_df = consolidated_busstate_df.drop(columns = ['EARLY_EVENT', 'LATE_EVENT', 'DEPARTURE_TIME', 'ENTER_STOP', 'EXIT_STOP'])

    # add hour and minute cols for filtering 
    consolidated_busstate_df['HOUR'] = consolidated_busstate_df['ARRIVAL'].dt.hour
    consolidated_busstate_df['MINUTE'] = consolidated_busstate_df['ARRIVAL'].dt.minute

    # ok - consolidated busstate df should now be ready for processing of the metrics.
    return consolidated_busstate_df

In [174]:
# This will produce df that metrics can be calculated from
mc_busstate_consolidated = process_mc_busstate()

## Calculate the headways for each stop

In [177]:
mc_busstate_consolidated

,BUS_ID,DATE,COUNT,STOP_ID,BOARDINGS,ALIGHTINGS,LOAD,RUN_ID,DEST,ARRIVAL,DEPARTURE,DWELL,HOUR,MINUTE
0,1301,2025-12-01,1,403,0,0,1,1512.0,MC,1900-01-01 05:50:04,1900-01-01 05:51:02,0 days 00:00:58,5,50
1,1301,2025-12-01,2,404,22,1,22,1512.0,MC,1900-01-01 05:52:01,1900-01-01 05:52:49,0 days 00:00:48,5,52
2,1301,2025-12-01,3,401,6,15,22,1512.0,MC,1900-01-01 05:57:46,1900-01-01 05:58:47,0 days 00:01:01,5,57
3,1301,2025-12-01,4,37,1,14,0,1512.0,MC,1900-01-01 05:59:21,1900-01-01 06:00:45,0 days 00:01:24,5,59
4,1301,2025-12-01,5,403,5,1,4,1512.0,MC,1900-01-01 06:07:08,1900-01-01 06:10:55,0 days 00:03:47,6,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36984,2503,2025-12-30,36964,404,1,9,1,1507.0,MC,1900-01-01 19:38:43,1900-01-01 19:40:51,0 days 00:02:08,19,38
36985,2503,2025-12-30,36965,94,0,2,0,1507.0,MC,1900-01-01 19:41:50,1900-01-01 19:42:17,0 days 00:00:27,19,41
36986,2503,2025-12-30,36966,95,0,0,0,1507.0,MC,1900-01-01 19:43:04,1900-01-01 19:43:31,0 days 00:00:27,19,43
36987,2503,2025-12-30,36967,401,0,0,0,1507.0,MC,1900-01-01 19:50:21,1900-01-01 19:50:33,0 days 00:00:12,19,50


In [188]:
# mc_busstate_consolidated['DWELL']
mc_busstate_consolidated['DATE'] = pd.to_datetime(mc_busstate_consolidated['DATE'], format = "%Y-%m-%d")
mc_busstate_consolidated.info()

<class 'pandas.DataFrame'>
RangeIndex: 36989 entries, 0 to 36988
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype          
---  ------      --------------  -----          
 0   BUS_ID      36989 non-null  int64          
 1   DATE        36989 non-null  datetime64[us] 
 2   COUNT       36989 non-null  int64          
 3   STOP_ID     36989 non-null  int64          
 4   BOARDINGS   36989 non-null  int64          
 5   ALIGHTINGS  36989 non-null  int64          
 6   LOAD        36989 non-null  int64          
 7   RUN_ID      36989 non-null  float64        
 8   DEST        36989 non-null  str            
 9   ARRIVAL     36989 non-null  datetime64[us] 
 10  DEPARTURE   36989 non-null  datetime64[us] 
 11  DWELL       36989 non-null  timedelta64[us]
 12  HOUR        36989 non-null  int32          
 13  MINUTE      36989 non-null  int32          
dtypes: datetime64[us](3), float64(1), int32(2), int64(6), str(1), timedelta64[us](1)
memory usage: 3.7 MB


In [200]:
def calculate_headway(busstate_df, stop_id):
    '''
    Calculate the headway for a given stop id in the busstate dataframe.

    Args:
        busstate_df (pd.DataFrame): The busstate dataframe containing the bus events.
        stop_id (int): The stop ID for which to calculate headways.
    Returns:
        pd.DataFrame: A subset dataframe containing the headways for the specified stop ID.
    '''
    # calculate headways of carmack 2 stop - 403
    stop_hw = busstate_df[busstate_df['STOP_ID'] == stop_id]

    stop_hw = stop_hw.sort_values(['DATE', 'ARRIVAL'])

    # stop_hw['ARRIVAL'].isna().sum() # zero NA
    stop_hw['HEADWAY'] = stop_hw['ARRIVAL'] - stop_hw['ARRIVAL'].shift(1) # headway in minutes

    stop_hw = stop_hw.loc[
        (stop_hw['HEADWAY'].dt.total_seconds() / 60 >= 0) & # filter out negative headways
        (stop_hw['HEADWAY'].dt.total_seconds() / 60 < 22) # filter out headways greater than 22 minutes - per legacy R code
        ] 

    # adjust for midnight arrivals where hour == 0
    stop_hw['DATE'] = stop_hw['DATE'].where(stop_hw['ARRIVAL'].dt.hour != 0,
                                                        stop_hw['DATE'] - pd.Timedelta(days = 1))
    stop_hw['HOUR'] = stop_hw['ARRIVAL'].dt.hour.where(stop_hw['ARRIVAL'].dt.hour != 0, 24)

    stop_hw.groupby(['DATE', 'ARRIVAL'])
    
    return stop_hw

In [201]:
carmack_2_id = 403
carmack_3_id = 404
university_hospital_id = 401
doan_hall_id = 37

carmack_2_hw = calculate_headway(mc_busstate_consolidated, carmack_2_id)
carmack_3_hw = calculate_headway(mc_busstate_consolidated, carmack_3_id)
university_hospital_hw = calculate_headway(mc_busstate_consolidated, university_hospital_id)
doan_hall_hw = calculate_headway(mc_busstate_consolidated, doan_hall_id)

In [ ]:
# combine all headways into one df for metrics calculation
combined_hw = pd.concat([carmack_2_hw, carmack_3_hw, university_hospital_hw, doan_hall_hw], ignore_index = True)

(combined_hw['HEADWAY'].dt.total_seconds() / 60).mean() # should be the average headway across all 4 stops for month of Dec in minutes.
(combined_hw['HEADWAY'].dt.total_seconds() / 60).median() # should be the median headway across all 4 stops for month of Dec in minutes.

np.float64(3.45)

In [220]:
combined_hw['HEADWAY'].describe()

count                     26226
mean     0 days 00:04:27.012049
std      0 days 00:04:00.855841
min             0 days 00:00:00
25%             0 days 00:01:44
50%             0 days 00:03:27
75%             0 days 00:06:00
max             0 days 00:21:59
Name: HEADWAY, dtype: object

In [222]:
combined_hw

,BUS_ID,DATE,COUNT,STOP_ID,BOARDINGS,ALIGHTINGS,LOAD,RUN_ID,DEST,ARRIVAL,DEPARTURE,DWELL,HOUR,MINUTE,HEADWAY
0,2302,2025-11-30,21326,403,0,0,0,1502.0,MC,1900-01-01 00:22:05,1900-01-01 00:25:27,0 days 00:03:22,24,22,0 days 00:00:00
1,2302,2025-11-30,21333,403,0,0,0,1502.0,MC,1900-01-01 00:41:50,1900-01-01 00:45:27,0 days 00:03:37,24,41,0 days 00:19:45
2,2302,2025-11-30,21334,403,0,0,0,1502.0,MC,1900-01-01 00:41:52,1900-01-01 00:45:19,0 days 00:03:27,24,41,0 days 00:00:02
3,2302,2025-12-01,21340,403,0,0,0,1502.0,MC,1900-01-01 01:00:57,1900-01-01 01:05:56,0 days 00:04:59,1,0,0 days 00:19:05
4,2302,2025-12-01,21341,403,0,0,0,1502.0,MC,1900-01-01 01:01:00,1900-01-01 01:05:48,0 days 00:04:48,1,1,0 days 00:00:03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26221,2103,2025-12-30,19089,37,1,0,3,1503.0,MC,1900-01-01 00:27:57,1900-01-01 00:28:43,0 days 00:00:46,24,27,0 days 00:21:39
26222,2104,2025-12-31,20163,37,0,25,0,1512.0,MC,1900-01-01 06:42:17,1900-01-01 06:43:17,0 days 00:01:00,6,42,0 days 00:20:10
26223,2104,2025-12-31,20167,37,0,16,7,1512.0,MC,1900-01-01 07:04:05,1900-01-01 07:05:00,0 days 00:00:55,7,4,0 days 00:21:48
26224,2104,2025-12-31,20173,37,13,12,19,1512.0,MC,1900-01-01 07:23:57,1900-01-01 07:27:29,0 days 00:03:32,7,23,0 days 00:19:52


In [ ]:
# # break down headway by each individual hour
# for row in combined_hw:
    